# Stage 1.3: The Prompt Registry
### Practical AgentOps: From PoC to Prod with MLflow 3 — ODSC AI East 2026

---

In the previous notebooks we hardcoded the system prompt as a Python string `SYSTEM_PROMPT`. We never changed the prompt, but our system prompt is extremely simple and will likely need refinement. Every time we iterated — adding measurement guidelines, dietary substitution language, safety caveats — we edited a cell and re-ran the notebook. That works during prototyping, but it has real problems:

- **No history** — which version of the prompt produced last Tuesday's eval results?
- **No separation of concerns** — prompt changes are mixed in with code changes
- **No deployment story** — how does a prompt go from "Sam likes this" to "production"?

MLflow's **Prompt Registry** solves all three. It gives you:

- **Versioned prompts** — every edit creates a new immutable version with a commit message
- **Aliases** — `@production`, `@staging`, `@latest` — so your app code never changes
- **Template variables** — `{{ style }}` syntax for parameterized prompts
- **Model config** — store temperature, max_tokens, and model name alongside the prompt
- **Automatic lineage** — loading a prompt inside a traced function links it to the trace

By the end of this notebook you will have:

- Registered the recipe assistant system prompt in the prompt registry
- Created a new version with improved guidelines
- Loaded prompts by version number and by alias
- Used template variables for a parameterized prompt
- Attached model config to a prompt
- Seen the prompt → trace lineage in the MLflow UI

---
## 0 — Environment Setup

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import mlflow

load_dotenv()

print(f"MLflow version: {mlflow.__version__}")
print(f"GEMINI_API_KEY present: {os.getenv('GEMINI_API_KEY') is not None}")

client = OpenAI(
    api_key=os.environ["GEMINI_API_KEY"],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

MODEL = "gemini-3.1-flash-lite-preview"

tracking_uri = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
mlflow.set_tracking_uri(tracking_uri)

experiment = mlflow.set_experiment("recipe-assistant")
mlflow.openai.autolog()

---
## 1 — The Problem: Prompts as Strings

Our system prompt in its current form:

```python
SYSTEM_PROMPT = """You are a helpful cooking assistant..."""

# After some feedback
SYSTEM_PROMPT_V2 = """You are a helpful cooking assistant...Please include estimated cook times."""
```

Questions this process can't answer:
- Which prompt version was running when User #347 got bad allergy advice?
- Did changing "be practical" to "be conversational" improve Correctness scores?
- Can we roll back to last week's prompt without a code deploy?

The prompt registry makes all of this trivial.

---
## 2 — Registering the System Prompt

`mlflow.genai.register_prompt()` creates a new prompt (or a new version of an existing one) in the registry. The prompt is stored in the tracking server's database and is immediately available to any notebook, script, or service that can reach the server.

Let's register the original system prompt from the first notebook — the bare-bones version before any of Sam's feedback.

In [ ]:
prompt_v1 = mlflow.genai.register_prompt(
    name="recipe-assistant-system",
    template=(
        "You are a helpful cooking assistant. Answer questions about recipes, "
        "cooking techniques, ingredients, and meal planning. Be practical and clear."
    ),
    commit_message="Initial system prompt — bare PoC version",
    tags={
        "author": "ml-team",
        "agent": "recipe-assistant",
        "stage": "poc",
    },
)

print(f"Prompt: {prompt_v1.name}")
print(f"Version: {prompt_v1.version}")
print(f"Template: {prompt_v1.template[:80]}...")

---
## 3 — Creating a New Version

When you call `register_prompt()` with an existing prompt name, MLflow creates a **new immutable version** rather than overwriting the old one. This means version 1 is always available for comparison or rollback.

Let's register the improved prompt that came out of Sam's product feedback — the one with measurements, timing, dietary substitutions, and safety caveats.

In [ ]:
prompt_v2 = mlflow.genai.register_prompt(
    name="recipe-assistant-system",
    template=(
        "You are a helpful cooking assistant for Mise en Place, a cooking app. "
        "Answer questions about recipes, cooking techniques, ingredients, and meal planning.\n\n"
        "Guidelines:\n"
        "- Always include specific measurements and time estimates\n"
        "- Be conversational and encouraging, like a knowledgeable friend\n"
        "- When giving recipes, briefly mention a vegetarian or dietary alternative if one exists\n"
        "- For food safety questions, be accurate and include appropriate caveats"
    ),
    commit_message="Added measurement, timing, dietary, and safety guidelines per product feedback",
    tags={
        "author": "ml-team",
        "agent": "recipe-assistant",
        "stage": "post-feedback",
    },
)

print(f"Version: {prompt_v2.version}")
print(f"Commit: {prompt_v2.commit_message}")

---
## 4 — Loading Prompts

`mlflow.genai.load_prompt()` retrieves a prompt from the registry. You can load by:

| Syntax | What it loads |
|---|---|
| `"prompts:/name/1"` | Specific version (immutable) |
| `"prompts:/name@latest"` | Most recent version |
| `"prompts:/name@production"` | Whatever version the `production` alias points to |
| `"name"` | Shorthand for `@latest` |

In [ ]:
# Load version 1 (the bare PoC prompt)
v1 = mlflow.genai.load_prompt("prompts:/recipe-assistant-system/1")
print("=== Version 1 ===")
print(v1.template)
print()

# Load version 2 (post-feedback prompt)
v2 = mlflow.genai.load_prompt("prompts:/recipe-assistant-system/2")
print("=== Version 2 ===")
print(v2.template)
print()

# Load latest (same as v2 right now)
latest = mlflow.genai.load_prompt("prompts:/recipe-assistant-system@latest")
print(f"Latest version: {latest.version}")

---
## 5 — Aliases: Decoupling Prompts from Code

Aliases are mutable labels that point to a specific version. Your application code loads `@production`, and you move the alias to a new version when you're ready — no code change, no redeploy.

A typical workflow:
1. Register a new prompt version
2. Run eval against it
3. If it passes, move the `@production` alias to that version
4. If something goes wrong, move `@production` back to the previous version

In [ ]:
# Point 'production' at version 1 (the safe, conservative prompt)
mlflow.genai.set_prompt_alias("recipe-assistant-system", alias="production", version=1)

# Point 'staging' at version 2 (the improved prompt we want to test)
mlflow.genai.set_prompt_alias("recipe-assistant-system", alias="staging", version=2)

print("Aliases set:")
print(f"  @production → version 1")
print(f"  @staging    → version 2")

In [ ]:
# Your app code only references the alias — never a version number
prod_prompt = mlflow.genai.load_prompt("prompts:/recipe-assistant-system@production")
staging_prompt = mlflow.genai.load_prompt("prompts:/recipe-assistant-system@staging")

print(f"Production (v{prod_prompt.version}): {prod_prompt.template[:60]}...")
print(f"Staging    (v{staging_prompt.version}): {staging_prompt.template[:60]}...")

In [ ]:
# After eval passes, promote staging to production — one line, no code deploy
mlflow.genai.set_prompt_alias("recipe-assistant-system", alias="production", version=2)

prod_prompt = mlflow.genai.load_prompt("prompts:/recipe-assistant-system@production")
print(f"Production now points to version {prod_prompt.version}")

---
## 6 — Using a Registered Prompt in the Agent

Now let's wire the registered prompt into our recipe agent. Instead of a hardcoded string, we load from the registry. This means the same agent code can run different prompt versions just by changing an alias.

In [ ]:
@mlflow.trace
def recipe_agent(inputs: dict) -> str:
    """Recipe assistant that loads its system prompt from the registry."""
    # Load the production prompt — automatically linked to this trace
    prompt = mlflow.genai.load_prompt("prompts:/recipe-assistant-system@production")

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": prompt.template},
            {"role": "user", "content": inputs["question"]},
        ],
        temperature=0.1,
        max_completion_tokens=512,
    )
    return response.choices[0].message.content


result = recipe_agent({"question": "How do I make scrambled eggs?"})
print(result)

### Prompt → Trace lineage

Open the MLflow UI and click on the trace that was just created. You'll see that the prompt version is **automatically linked** to the trace — no extra code needed. `load_prompt()` inside a traced function does this for you.

This answers the question: "Which prompt version was running when this trace was generated?"

---
## 7 — Template Variables

So far our system prompt is a static string. But what if you want the same prompt to work differently for different contexts? Template variables let you parameterize parts of the prompt using `{{ variable }}` syntax.

For our cooking assistant, imagine we want to adjust the style based on the user's experience level — beginners get more detail, experienced cooks get concise answers.

In [ ]:
prompt_v3 = mlflow.genai.register_prompt(
    name="recipe-assistant-system",
    template=(
        "You are a helpful cooking assistant for Mise en Place, a cooking app. "
        "Answer questions about recipes, cooking techniques, ingredients, and meal planning.\n\n"
        "The user's experience level is: {{ experience_level }}\n\n"
        "Adapt your responses accordingly:\n"
        "- For beginners: explain techniques in detail, define culinary terms, suggest easier alternatives\n"
        "- For intermediate: focus on tips and common mistakes to avoid\n"
        "- For advanced: be concise, use professional terminology, suggest refinements\n\n"
        "Guidelines:\n"
        "- Always include specific measurements and time estimates\n"
        "- Be conversational and encouraging, like a knowledgeable friend\n"
        "- When giving recipes, briefly mention a vegetarian or dietary alternative if one exists\n"
        "- For food safety questions, be accurate and include appropriate caveats"
    ),
    commit_message="Added experience_level template variable for adaptive responses",
    tags={
        "author": "ml-team",
        "agent": "recipe-assistant",
        "stage": "personalization",
    },
)

print(f"Version: {prompt_v3.version}")
print(f"Template preview: {prompt_v3.template[:100]}...")

In [ ]:
# Load and format the prompt with a variable
prompt = mlflow.genai.load_prompt("prompts:/recipe-assistant-system/3")

# .format() fills in the template variables
beginner_prompt = prompt.format(experience_level="beginner")
advanced_prompt = prompt.format(experience_level="advanced")

print("=== Beginner ===")
print(beginner_prompt[:200])
print()
print("=== Advanced ===")
print(advanced_prompt[:200])

In [ ]:
@mlflow.trace
def recipe_agent_adaptive(inputs: dict) -> str:
    """Recipe assistant with experience-level adaptation."""
    prompt = mlflow.genai.load_prompt("prompts:/recipe-assistant-system/3")
    experience = inputs.get("experience_level", "intermediate")
    system_content = prompt.format(experience_level=experience)

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_content},
            {"role": "user", "content": inputs["question"]},
        ],
        temperature=0.1,
        max_completion_tokens=512,
    )
    return response.choices[0].message.content


# Same question, different experience levels
print("=== Beginner ===")
print(recipe_agent_adaptive({"question": "How do I deglaze a pan?", "experience_level": "beginner"}))
print()
print("=== Advanced ===")
print(recipe_agent_adaptive({"question": "How do I deglaze a pan?", "experience_level": "advanced"}))

---
## 8 — Chat-Style Prompt Templates

So far we've registered text prompts — single strings. But the OpenAI SDK takes a list of messages. You can register **chat-style templates** that include the role structure, so the entire message format is versioned together.

In [ ]:
chat_prompt = mlflow.genai.register_prompt(
    name="recipe-assistant-chat",
    template=[
        {
            "role": "system",
            "content": (
                "You are a helpful cooking assistant for Mise en Place. "
                "Respond to {{ experience_level }} cooks with appropriate detail.\n\n"
                "Guidelines:\n"
                "- Always include specific measurements and time estimates\n"
                "- Be conversational and encouraging"
            ),
        },
        {
            "role": "user",
            "content": "{{ question }}",
        },
    ],
    commit_message="Chat-style template with system and user messages",
    tags={"author": "ml-team", "format": "chat"},
)

print(f"Prompt: {chat_prompt.name} (version {chat_prompt.version})")
print(f"Template type: {type(chat_prompt.template)}")

---
## 9 — Storing Model Config with the Prompt

A prompt is only half the story — the model, temperature, and max tokens matter too. You can store **model config** alongside the prompt so that the full generation configuration is versioned together.

This is especially useful when you're testing different model/prompt combinations: "version 2 with gpt-4o at temp 0.3" vs "version 2 with gemini-flash at temp 0.1".

In [ ]:
prompt_with_config = mlflow.genai.register_prompt(
    name="recipe-assistant-configured",
    template=(
        "You are a helpful cooking assistant for Mise en Place, a cooking app. "
        "Answer questions about recipes, cooking techniques, ingredients, and meal planning.\n\n"
        "Guidelines:\n"
        "- Always include specific measurements and time estimates\n"
        "- Be conversational and encouraging, like a knowledgeable friend\n"
        "- When giving recipes, briefly mention a vegetarian or dietary alternative if one exists\n"
        "- For food safety questions, be accurate and include appropriate caveats"
    ),
    model_config={
        "model_name": "gemini-3.1-flash-lite-preview",
        "temperature": 0.1,
        "max_tokens": 512,
    },
    commit_message="System prompt with model config for Gemini Flash Lite",
    tags={"author": "ml-team", "provider": "gemini"},
)

print(f"Version: {prompt_with_config.version}")
print(f"Model config: {prompt_with_config.model_config}")

In [ ]:
# Load the prompt and use its model config for the API call
prompt = mlflow.genai.load_prompt("recipe-assistant-configured")

response = client.chat.completions.create(
    model=prompt.model_config["model_name"],
    messages=[
        {"role": "system", "content": prompt.template},
        {"role": "user", "content": "How do I make a roux?"},
    ],
    temperature=prompt.model_config["temperature"],
    max_completion_tokens=prompt.model_config["max_tokens"],
)

print(response.choices[0].message.content)

---
## 10 — Searching and Managing Prompts

As your prompt library grows, you need to find and organize prompts. The registry supports search, tagging, and deletion.

In [ ]:
# Find all prompts for the recipe assistant
prompts = mlflow.genai.search_prompts(filter_string="agent='recipe-assistant'")

print(f"Found {len(prompts)} prompts:")
for p in prompts:
    print(f"  {p.name} — {p.version} version(s), tags: {p.tags}")

In [ ]:
# Add tags at the prompt level (applies to all versions)
mlflow.genai.set_prompt_tag("recipe-assistant-system", "team", "mise-en-place")
mlflow.genai.set_prompt_tag("recipe-assistant-system", "status", "active")

# Add tags at the version level (specific to one version)
mlflow.genai.set_prompt_version_tag("recipe-assistant-system", version=2, key="eval-status", value="passed")

# Check tags
tags = mlflow.genai.get_prompt_tags("recipe-assistant-system")
print(f"Prompt-level tags: {tags}")

v2 = mlflow.genai.load_prompt("prompts:/recipe-assistant-system/2")
print(f"Version 2 tags: {v2.tags}")

---
## 11 — Putting It All Together: The Prompt Iteration Workflow

Here's how the prompt registry fits into the development cycle:

```
1. Register prompt v1                    → mlflow.genai.register_prompt()
2. Set @production alias to v1           → mlflow.genai.set_prompt_alias()
3. Agent loads @production in app code   → mlflow.genai.load_prompt("...@production")
4. Product feedback arrives              → iterate on the prompt text
5. Register prompt v2 (same name)        → mlflow.genai.register_prompt()
6. Set @staging alias to v2              → mlflow.genai.set_prompt_alias()
7. Run eval against @staging             → mlflow.genai.evaluate()
8. If eval passes, promote to production → mlflow.genai.set_prompt_alias()
9. If something breaks, rollback         → mlflow.genai.set_prompt_alias() back to v1
```

The app code never changes — it always loads `@production`. The prompt registry handles the rest.

In [ ]:
# This is what your production agent looks like — clean and decoupled

@mlflow.trace
def production_recipe_agent(inputs: dict) -> str:
    """Production agent — prompt version is controlled by the registry, not by code."""
    prompt = mlflow.genai.load_prompt("prompts:/recipe-assistant-system@production")

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": prompt.template},
            {"role": "user", "content": inputs["question"]},
        ],
        temperature=0.1,
        max_completion_tokens=512,
    )
    return response.choices[0].message.content


# Test it
result = production_recipe_agent({"question": "How do I make chicken stock?"})
print(result[:300], "...")

---
## Summary: What We Built

| Concept | How we used it |
|---|---|
| `register_prompt()` | Registered the system prompt with versioning and commit messages |
| Immutable versions | Created v1 (PoC), v2 (post-feedback), v3 (with template variables) |
| `load_prompt()` | Retrieved prompts by version number, by alias, and by `@latest` |
| Aliases | Set `@production` and `@staging`, promoted staging to production |
| `{{ variable }}` syntax | Parameterized the prompt with `experience_level` |
| `.format()` | Filled in template variables at runtime |
| Chat-style templates | Registered a multi-message prompt with role structure |
| `model_config` | Stored model name, temperature, and max_tokens with the prompt |
| Prompt → trace lineage | `load_prompt()` inside `@mlflow.trace` auto-links prompt to trace |
| `search_prompts()` | Found prompts by tag |
| Prompt and version tags | Organized prompts with metadata at both levels |

### Why this matters

| Without the registry | With the registry |
|---|---|
| Prompt changes are code changes | Prompt changes are registry operations |
| Rollback = git revert + redeploy | Rollback = move an alias |
| "Which prompt was running?" = grep through logs | Prompt version linked to every trace |
| A/B testing = feature flags | A/B testing = two aliases |
| Team collaboration = merge conflicts | Team collaboration = versioned registry |

---

**Next →** With prompts versioned and eval datasets stored, you have two of the three pillars for a production-grade evaluation pipeline. In the next notebook we'll connect them: load a prompt from the registry, run it against the stored eval dataset, and track the results — all with full lineage.